In [114]:
import geopandas as gpd
from pyproj import Transformer
from requests import get, post
import pandas as pd
from shapely.geometry import Polygon, Point, MultiPolygon
import numpy as np

In [73]:
zoning_map = get('https://mapsa.slocity.org/hosting/rest/services/sloGISlayers/gisLandManagementWM/MapServer/5/query?where=1%3D1&f=pjson').json()


zones = pd.json_normalize(zoning_map['features'])
zones = zones.explode(column='geometry.rings')


arcgis_to_lat_log = Transformer.from_crs("EPSG:3857", "EPSG:4326")

zones['polygon'] = zones['geometry.rings'].map(lambda x: [arcgis_to_lat_log.transform(point[0], point[1])[::-1] for point in x])

zones = gpd.GeoDataFrame(zones['attributes.generalZone'].rename('zone'), geometry=zones['polygon'].map(Polygon))

del zoning_map
zones

,zone,geometry
0,R-1,"POLYGON ((-120.62803 35.26558, -120.62803 35.2..."
1,C/OS,"POLYGON ((-120.69361 35.31137, -120.69368 35.3..."
2,R-1,"POLYGON ((-120.64883 35.29331, -120.64884 35.2..."
3,R-1,"POLYGON ((-120.64659 35.29492, -120.64658 35.2..."
4,R-1,"POLYGON ((-120.64769 35.29465, -120.64769 35.2..."
...,...,...
1320,R-1,"POLYGON ((-120.63247 35.25076, -120.63251 35.2..."
1321,PF,"POLYGON ((-120.63378 35.2519, -120.63386 35.25..."
1322,R-1,"POLYGON ((-120.63309 35.25242, -120.63318 35.2..."
1323,PF,"POLYGON ((-120.63377 35.25326, -120.63381 35.2..."


In [17]:
license_data = pd.read_csv('./cleaned.csv', index_col=0)

In [26]:
def get_zone(point : Point):
    return None if point == Point(0,0) else zones['zone'][zones.distance(point).idxmin()]

get_zone(Point(license_data['longitude'][0], license_data['latitude'][0]))

'C-D'

In [28]:
license_data[['longitude','latitude']].apply(lambda x: Point(x),axis=1).map(get_zone)

0       C-D
1       R-2
2       R-1
3       C-S
4       R-1
       ... 
8760    C-D
8761    R-2
8762      M
8763    C-S
8764    C-R
Length: 8765, dtype: object

In [138]:
offset = 0

data = pd.DataFrame()
while True:
    offset = len(data)
    new_data = pd.json_normalize(get(f'https://mapsa.slocity.org/hosting/rest/services/sloGISlayers/gisLandManagementWM/MapServer/4/query?where=1%3D1&f=pjson&resultOffset={offset}').json()['features'])
    data  = pd.concat([data, new_data]).reset_index(drop=True)
    if new_data.empty:
        break

parcels = gpd.GeoDataFrame(geometry=data['geometry.rings'].map(lambda x: [arcgis_to_lat_log.transform(point[0], point[1])[::-1] for point in x[0]]).map(Polygon)) # Convert points to shape objects

parcels['APN'] = data['attributes.APN']

del new_data
del data

def get_parcel(point : Point):
    return None if point == Point(0,0) else parcels['APN'][parcels.distance(point).idxmin()]

parcels

,geometry,APN
0,"POLYGON ((-120.67176 35.28901, -120.67176 35.2...",001-012-006
1,"POLYGON ((-120.67206 35.28854, -120.67229 35.2...",001-012-008
2,"POLYGON ((-120.67206 35.28854, -120.67212 35.2...",001-012-009
3,"POLYGON ((-120.67255 35.29033, -120.67255 35.2...",001-012-012
4,"POLYGON ((-120.67234 35.29033, -120.67234 35.2...",001-012-013
...,...,...
17326,"POLYGON ((-120.64412 35.26909, -120.64428 35.2...",004-785-011
17327,"POLYGON ((-120.64347 35.26948, -120.64373 35.2...",004-785-001
17328,"POLYGON ((-120.66348 35.27232, -120.66348 35.2...",003-623-002
17329,"POLYGON ((-120.64737 35.29002, -120.64757 35.2...",001-251-034


In [152]:
get_parcel(Point(license_data['longitude'][0], license_data['latitude'][0]))

'002-423-013'

In [164]:
type_mappings = {
    "Downtown Association" : "Downtown Association",
    "Residential Rental" : "Rental",
    "Commercial Rental Property" : "Rental",
    "General Service" : "Service",
    "Professional Service" :"Service",
    "General Retailer" : "Retail",
    "Contractor" : "Service",
    "Massage Therapist" : "Service",
    "General Manufacturer" : "Other",
    "General Wholesaler" : "Other",
    "Homestay Temporary TOT" : "Hotel",
    "Utilities/Transportation" : "Other",
    "Cannabis Permit Fee" : "Retail",
    "General Gun Dealer" : "Retail",
    "Cannabis Delivery Driver" : "Service"
}


license_data['type'] = license_data['Rate type (STD)'].map(type_mappings)

del type_mappings
license_data

,DBA,Business type,Bus address,Start date,Close date,License description,Rate type (STD),closure_time,censor,type,longitude,latitude
0,!ROMP,Apparel/Accessories,714 HIGUERA ST,4/6/2005,5/3/2019,NaN,Downtown Association,7638,1,Downtown Association,-120.664277,35.279094
1,"""""""Angel Lynn""""""",Misc Business Services,3860 S HIGUERA ST SP A24,6/8/2009,1/1/2012,"Face Painting, Balloon Sculpture, Card Tricks,...",General Service,6114,1,Service,-120.673262,35.247583
2,"""""""Slack""""""",Rental- Residential,2045 SLACK ST,10/28/2009,4/20/2017,Residential Property Rental,Residential Rental,5972,1,Rental,-120.650013,35.295909
3,{Ther-Happy} By Natalie Garay,Miscellaneous Business,4251 S HIGUERA ST STE 300,3/4/2019,8/1/2020,Private Pilates Instrucion & Flower Essence Th...,General Service,2558,1,Service,-120.676363,35.241632
4,1 Dream Education,Home Occupation,357 SAGE ST,5/5/2015,5/31/2022,Education Consulting,Professional Service,3957,1,Service,-120.662626,35.255449
...,...,...,...,...,...,...,...,...,...,...,...,...
8760,Zoi Jewelry/Le Monarque,Retail,1075 COURT ST STE 202,2/25/2014,10/9/2023,"Antique, Estate & Vintage Jewelry, Resale Online",Downtown Association,4391,1,Downtown Association,-120.661469,35.281076
8761,Zoric Logistics LLC,Home Occupation,994 BLUEBELL WAY,10/6/2021,8/11/2025,Transportation & Delivery of Goods,General Service,1611,1,Service,-120.636923,35.249175
8762,Zs Unlimited,Auto Mechanic/Detailer/Services,3621 SACRAMENTO DR 6,6/25/1993,11/27/2007,NaN,General Service,11941,1,Service,-120.641378,35.254683
8763,Zumer Products Llc,"Wholesale, Distributors",3563 SUELDO ST Q,8/10/2000,10/21/2009,NaN,General Wholesaler,9338,1,Other,-120.665576,35.251453


In [169]:
license_data.to_csv('./cleaned.csv', index=False)
